# ML Model with actual goals

In [67]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
#from functools import cache
import warnings

NUM_OF_COUNTRIES = 5
NUM_OF_SEASONS = 25
NUM_OF_MATCHES = 9130
SEED = 1893

np.random.seed(SEED)
warnings.filterwarnings('ignore')


In [68]:
def score_from_amount_and_diff(amount, diff):
    if diff < 0:
        home_score = (amount + diff) / 2
        away_score = home_score - diff
    else:
        away_score = (amount - diff) / 2
        home_score = away_score + diff

    # cut excess floating numbers
    if (home_score - int(home_score)) == 0 and (away_score - int(away_score)) == 0:
        home_score = int(home_score)
        away_score = int(away_score)

    return home_score, away_score


def closed_form_solution(X, y):
    # alternative: np.dot(np.linalg.inv(np.dot(X.T, X)), np.dot(X.T, y))
    return np.linalg.solve(np.dot(X.T, X), np.dot(X.T, y))


def h(X, w):
    return np.matmul(X, w)[0]


In [69]:
df_matches = pd.read_csv('matches_ac.csv', sep=';')
df_matches['bias'] = 1

noise = np.random.normal(0, 0.01, df_matches.shape[0] * 10)

for index, row in df_matches.iterrows():
    # ht
    if row["ht_curr_eff_chancediff"] == 0:
        df_matches.at[index, "ht_curr_eff_chancediff"] = noise[index]
    if row["ht_past_chancediff_combined"] == 0:
        df_matches.at[index,
                      "ht_past_chancediff_combined"] = noise[df_matches.shape[0] + index]
    if row["ht_curr_eff_chanceamount"] == 0:
        df_matches.at[index,
                      "ht_curr_eff_chanceamount"] = noise[df_matches.shape[0] * 2 + index]
    if row["ht_past_chanceamount"] == 0:
        df_matches.at[index,
                      "ht_past_chanceamount"] = noise[df_matches.shape[0] * 3 + index]
    # at
    if row["at_curr_eff_chancediff"] == 0:
        df_matches.at[index,
                      "at_curr_eff_chancediff"] = noise[df_matches.shape[0] * 4 + index]
    if row["at_past_chancediff_combined"] == 0:
        df_matches.at[index,
                      "at_past_chancediff_combined"] = noise[df_matches.shape[0] * 5 + index]
    if row["at_curr_eff_chanceamount"] == 0:
        df_matches.at[index,
                      "at_curr_eff_chanceamount"] = noise[df_matches.shape[0] * 6 + index]
    if row["at_past_chanceamount"] == 0:
        df_matches.at[index,
                      "at_past_chanceamount"] = noise[df_matches.shape[0] * 7 + index]
    # label
    if row["actual_goaldiff"] == 0:
        df_matches.at[index,
                      "actual_goaldiff"] = noise[df_matches.shape[0] * 8 + index]
    if row["actual_goalamount"] == 0:
        df_matches.at[index,
                      "actual_goalamount"] = noise[df_matches.shape[0] * 9 + index]

df_matches


,country,season,matchnumber,ht_name,at_name,ht_curr_eff_chancediff,at_curr_eff_chancediff,ht_past_chancediff_combined,at_past_chancediff_combined,ht_curr_eff_chanceamount,at_curr_eff_chanceamount,ht_past_chanceamount,at_past_chanceamount,ht_market_value_ln_relative,at_market_value_ln_relative,ht_is_promoter,at_is_promoter,actual_goalamount,actual_goaldiff,bias
0,ENG,14,1,Queens Park Rangers,Hull,-0.000107,-0.006322,0.005476,0.002399,-0.006037,0.012542,0.006067,-0.005316,-0.392415,-0.348740,True,False,1,-1,1
1,ENG,14,1,Newcastle,Man City,-0.002312,0.004580,0.002748,0.007405,0.008663,0.001433,-0.000951,-0.003893,-0.169272,1.002768,False,False,2,-2,1
2,ENG,14,1,Burnley,Chelsea,-0.004361,-0.000947,-0.009217,-0.020822,-0.010237,0.010523,-0.005959,0.010205,-1.140622,1.211651,True,False,4,-2,1
3,ENG,14,1,Arsenal,Crystal Palace,0.007917,0.024501,0.008286,0.010394,0.001439,0.007554,-0.013947,0.000835,0.917365,-0.536203,False,False,3,1,1
4,ENG,14,1,West Ham,Tottenham,0.004695,-0.004871,0.002485,-0.000451,0.001249,0.018424,0.008236,0.009088,-0.198259,0.606114,False,False,1,-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9125,ITA,18,38,Frosinone,Chievo,-4.492426,-4.452152,-3.581579,-1.943421,-2.059192,-2.232642,4.228070,4.508772,-1.394930,-1.394930,True,False,0,0,1
9126,ITA,18,38,Fiorentina,CFC Genoa,1.172165,-1.286395,1.539474,-1.361842,0.407509,-0.820735,6.789474,4.719298,0.350310,-0.144768,False,False,0,0,1
9127,ITA,18,38,Bologna,Napoli,-0.932129,5.242267,-1.497368,5.293421,-0.446689,3.415385,4.350877,8.877193,-0.614771,1.070083,False,False,5,1,1
9128,ITA,18,38,Inter,Empoli,2.881676,-3.462648,2.380263,-3.058772,1.042860,-0.805584,6.754386,4.403509,1.024225,-0.738824,False,True,3,1,1


## general weights

In [70]:



training_data = df_matches.query(f"season <= 17")

X_training_data = training_data[["ht_curr_eff_chancediff", "at_curr_eff_chancediff", "ht_past_chancediff_combined", "at_past_chancediff_combined",
                                    "ht_curr_eff_chanceamount", "at_curr_eff_chanceamount", "ht_past_chanceamount", "at_past_chanceamount", "ht_market_value_ln_relative", "at_market_value_ln_relative", "bias"]].values
y_training_data_ga = training_data[["actual_goalamount"]].values
y_training_data_gd = training_data[["actual_goaldiff"]].values

w_ga = closed_form_solution(X_training_data, y_training_data_ga)
w_gd =  closed_form_solution(X_training_data, y_training_data_gd)

country, season, matchnumber, ht_name, at_name = "ENG", 18, 20, "Burnley", "West Ham"

prospect_game = df_matches.query(
    f"country == '{country}' and season == {season} and matchnumber == {matchnumber} and ht_name == '{ht_name}' and at_name == '{at_name}'")
input_X = prospect_game[["ht_curr_eff_chancediff", "at_curr_eff_chancediff", "ht_past_chancediff_combined", "at_past_chancediff_combined",
                         "ht_curr_eff_chanceamount", "at_curr_eff_chanceamount", "ht_past_chanceamount", "at_past_chanceamount", "ht_market_value_ln_relative", "at_market_value_ln_relative", "bias"]].values

goalamount, goaldiff = h(input_X, w_ga), h(input_X, w_gd)

ht_score, at_score = score_from_amount_and_diff(goalamount, goaldiff)
print(ht_name, at_name, str(ht_score)[0:4]+":"+str(at_score)[0:4])


Burnley West Ham [1.1:[1.3


## weights per matchnumber

In [71]:
ga_weights, gd_weights = [], []
for i in range(1,39):
    training_data = df_matches.query(f"season <= 17 and matchnumber == {i}")

    X_training_data = training_data[["ht_curr_eff_chancediff", "at_curr_eff_chancediff", "ht_past_chancediff_combined", "at_past_chancediff_combined",
                                    "ht_curr_eff_chanceamount", "at_curr_eff_chanceamount", "ht_past_chanceamount", "at_past_chanceamount", "ht_market_value_ln_relative", "at_market_value_ln_relative", "bias"]].values
    y_training_data_ga = training_data[["actual_goalamount"]].values
    y_training_data_gd = training_data[["actual_goaldiff"]].values

    ga_weights.append(closed_form_solution(X_training_data, y_training_data_ga))
    gd_weights.append(closed_form_solution(X_training_data, y_training_data_gd))

country, season, matchnumber, ht_name, at_name = "ENG", 18, 20, "Burnley", "West Ham"

prospect_game = df_matches.query(
    f"country == '{country}' and season == {season} and matchnumber == {matchnumber} and ht_name == '{ht_name}' and at_name == '{at_name}'")
input_X = prospect_game[["ht_curr_eff_chancediff", "at_curr_eff_chancediff", "ht_past_chancediff_combined", "at_past_chancediff_combined",
                         "ht_curr_eff_chanceamount", "at_curr_eff_chanceamount", "ht_past_chanceamount", "at_past_chanceamount", "ht_market_value_ln_relative", "at_market_value_ln_relative", "bias"]].values

goalamount, goaldiff = h(
    input_X, ga_weights[matchnumber-1]), h(input_X, gd_weights[matchnumber-1])

ht_score, at_score = score_from_amount_and_diff(goalamount, goaldiff)
print(ht_name, at_name, str(ht_score)[0:4]+":"+str(at_score)[0:4])


Burnley West Ham [1.1:[1.8


## error general weights

In [72]:
test_data_df = df_matches.query(f"season == 18")

ga_squared_error, gd_squared_error = 0, 0

for index, row in test_data_df.iterrows():
    input_X = row[["ht_curr_eff_chancediff", "at_curr_eff_chancediff", "ht_past_chancediff_combined", "at_past_chancediff_combined",
                   "ht_curr_eff_chanceamount", "at_curr_eff_chanceamount", "ht_past_chanceamount", "at_past_chanceamount", "ht_market_value_ln_relative", "at_market_value_ln_relative", "bias"]].values
    actual_ga, actual_gd = row["actual_goalamount"], row["actual_goaldiff"]
    ga_squared_error += np.square(h(input_X, w_ga) - actual_ga)
    gd_squared_error += np.square(h(input_X, w_gd) - actual_gd)

print("Squared error goalamount", ga_squared_error / 1826)
print("Squared error goaldiff", gd_squared_error / 1826)


Squared error goalamount 2.7042974941273674
Squared error goaldiff 2.6148814894738868


## error weights per matchnumber

In [73]:
ga_squared_error, gd_squared_error = 0, 0

for index, row in test_data_df.iterrows():
    input_X = row[["ht_curr_eff_chancediff", "at_curr_eff_chancediff", "ht_past_chancediff_combined", "at_past_chancediff_combined",
                   "ht_curr_eff_chanceamount", "at_curr_eff_chanceamount", "ht_past_chanceamount", "at_past_chanceamount", "ht_market_value_ln_relative", "at_market_value_ln_relative", "bias"]].values
    actual_ga, actual_gd = row["actual_goalamount"], row["actual_goaldiff"]
    ga_squared_error += np.square(h(input_X,
                                    ga_weights[matchnumber-1]) - actual_ga)
    gd_squared_error += np.square(h(input_X,
                                    gd_weights[matchnumber-1]) - actual_gd)

print("Squared error goalamount", ga_squared_error / 1826)
print("Squared error goaldiff", gd_squared_error / 1826)


Squared error goalamount 3.0967407091734844
Squared error goaldiff 2.720118342493738
